In [1]:
import os, gc, time, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
import rasterio.transform
from rasterio.warp import transform as warp_transform
from scipy import ndimage
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_ROOT       = Path("/home/23CSU413/Deep Learning/AgriFieldNet")
SOURCE_DIR      = DATA_ROOT / "source"
TRAIN_LABEL_DIR = DATA_ROOT / "train_labels" / "train_labels"
TEST_LABEL_DIR  = DATA_ROOT / "test_labels"
CACHE = Path("cache"); CACHE.mkdir(exist_ok=True)

BANDS = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"]
CROP_MAP = {1:"Wheat",2:"Mustard",3:"Lentil",4:"Fallow",5:"GreenPea",6:"Sugarcane",
            8:"Garlic",9:"Maize",13:"Gram",14:"Coriander",15:"Potato",16:"Berseem",36:"Rice"}
ORIG_LABELS = sorted(CROP_MAP)
LABEL2IDX   = {c: i for i, c in enumerate(ORIG_LABELS)}
CLASS_NAMES = [CROP_MAP[c] for c in ORIG_LABELS]
K = len(ORIG_LABELS)          # 13 classes
PATCH = 32                    # CNN patch size
USE_LOCATION = True           # use field lon/lat as features (set False to test without)

def chip_ids_from(folder):
    return sorted({p.stem.split("_")[-3] for p in folder.glob("*_field_ids.tif")})

all_chip_ids   = sorted(p.name.split("_")[-1] for p in SOURCE_DIR.iterdir() if p.is_dir())
train_chip_ids = chip_ids_from(TRAIN_LABEL_DIR)
test_chip_ids  = chip_ids_from(TEST_LABEL_DIR)
print("source chips:", len(all_chip_ids), "| train-label chips:", len(train_chip_ids),
      "| test-label chips:", len(test_chip_ids))

source chips: 1217 | train-label chips: 1165 | test-label chips: 707


In [2]:
import requests, rasterio
from rasterio.warp import transform_bounds
import pystac_client          # imported fine in your run; no pydantic needed

STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"

# 1) Can this server reach Planetary Computer (search + data-token endpoints)?
for name, url in [("STAC API", STAC),
                  ("SAS token", "https://planetarycomputer.microsoft.com/api/sas/v1/token/sentinel-2-l2a")]:
    try:
        r = requests.get(url, timeout=20)
        print(f"{name}: HTTP {r.status_code}")
    except Exception as e:
        print(f"{name}: NO ACCESS -> {e}")

# 2) Scenes covering one chip
cid = train_chip_ids[0]
tif = SOURCE_DIR / f"ref_agrifieldnet_competition_v1_source_{cid}" / f"ref_agrifieldnet_competition_v1_source_{cid}_B02_10m.tif"
with rasterio.open(tif) as src:
    bbox = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
print("chip bbox (lon/lat):", bbox)

catalog = pystac_client.Client.open(STAC)          # no signing needed for searching
s2 = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime="2021-10-01/2022-05-31",
                    query={"eo:cloud_cover": {"lt": 40}}).item_collection()
print("Sentinel-2 scenes (<40% cloud):", len(s2))
print(sorted({i.datetime.date().isoformat() for i in s2}))

s1 = catalog.search(collections=["sentinel-1-rtc"], bbox=bbox,
                    datetime="2021-10-01/2022-05-31").item_collection()
print("Sentinel-1 RTC scenes:", len(s1))
print(sorted({i.datetime.date().isoformat() for i in s1})[:10], "...")

STAC API: HTTP 200
SAS token: HTTP 200
chip bbox (lon/lat): (87.89243132681646, 25.61998576133954, 87.91810354240825, 25.643258474933948)
Sentinel-2 scenes (<40% cloud): 38
['2021-10-11', '2021-11-05', '2021-11-10', '2021-11-15', '2021-11-30', '2021-12-05', '2021-12-10', '2021-12-15', '2021-12-20', '2021-12-25', '2022-01-29', '2022-02-03', '2022-02-08', '2022-02-13', '2022-02-18', '2022-02-23', '2022-02-28', '2022-03-05', '2022-03-10', '2022-03-15', '2022-03-20', '2022-03-25', '2022-04-04', '2022-04-24', '2022-05-04', '2022-05-09', '2022-05-29']
Sentinel-1 RTC scenes: 39
['2021-10-05', '2021-10-08', '2021-10-17', '2021-10-20', '2021-10-29', '2021-11-01', '2021-11-10', '2021-11-13', '2021-11-22', '2021-11-25'] ...


In [3]:
import time, threading
import numpy as np, pandas as pd, requests, rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling
from scipy import ndimage
import pystac_client
from concurrent.futures import ThreadPoolExecutor, as_completed

TS_DIR = CACHE / "ts_fast"; TS_DIR.mkdir(exist_ok=True)
STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
DATE_RANGE = "2021-10-01/2022-05-31"
S2_ASSETS = ["B03", "B04", "B05", "B08", "B11", "SCL"]
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff,.TIF,.TIFF",
                GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="1")

# ---- manual SAS-token signing (replaces planetary_computer.sign) ----
_tok, _tok_t, _lock = {}, {}, threading.Lock()
def sas(collection):
    with _lock:
        if collection not in _tok or time.time() - _tok_t[collection] > 2400:   # refresh every 40 min
            r = requests.get(f"https://planetarycomputer.microsoft.com/api/sas/v1/token/{collection}", timeout=30)
            r.raise_for_status()
            _tok[collection], _tok_t[collection] = r.json()["token"], time.time()
        return _tok[collection]

def read_asset(item, key, collection, bounds, crs, shape, resampling):
    """Read one asset resampled onto the chip's 256x256 grid."""
    href = item.assets[key].href + "?" + sas(collection)
    with rasterio.Env(**GDAL_ENV):
        with rasterio.open(href) as src:
            b = transform_bounds(crs, src.crs, *bounds)
            win = from_bounds(*b, transform=src.transform)
            return src.read(1, window=win, out_shape=shape, resampling=resampling,
                            boundless=True, fill_value=0).astype(np.float32)

def chip_fields(cid):
    """[(split, field_id, flat pixel indices of the field's inner 'core')] for one chip."""
    out = []
    for split, folder, tag in [("train", TRAIN_LABEL_DIR, "labels_train"), ("test", TEST_LABEL_DIR, "labels_test")]:
        p = folder / f"ref_agrifieldnet_competition_v1_{tag}_{cid}_field_ids.tif"
        if not p.exists(): continue
        with rasterio.open(p) as src: fld = src.read(1).astype(np.int64)
        for fid in np.unique(fld):
            if fid == 0: continue
            m = fld == fid
            er = ndimage.binary_erosion(m, border_value=1)
            if er.sum() < 3: er = m
            out.append((split, int(fid), np.flatnonzero(er.ravel())))
    return out

In [4]:
EPS = 1e-6
def search(cat, coll, bbox, **kw):
    for attempt in range(3):
        try: return list(cat.search(collections=[coll], bbox=bbox, datetime=DATE_RANGE, **kw).items())
        except Exception: time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"STAC search failed for {coll}")

In [5]:
import threading
from collections import Counter

TS_DIR = CACHE / "ts_fast"; TS_DIR.mkdir(exist_ok=True)       # new folder, old pilot data untouched
S2_ASSETS = ["B03", "B04", "B08", "B11", "SCL"]

def thin(items, min_gap):
    kept, last = [], None
    for it in sorted(items, key=lambda i: i.datetime):
        d = it.datetime.date()
        if last is None or (d - last).days >= min_gap:
            kept.append(it); last = d
    return kept

def process_chip(cid):
    out_f = TS_DIR / f"{cid}.pkl"
    if out_f.exists(): return 0
    fields = chip_fields(cid)
    if not fields:
        pd.to_pickle(pd.DataFrame(), out_f); return 0
    tif = SOURCE_DIR / f"ref_agrifieldnet_competition_v1_source_{cid}" / f"ref_agrifieldnet_competition_v1_source_{cid}_B02_10m.tif"
    with rasterio.open(tif) as src:
        bounds, crs, shape = src.bounds, src.crs, (src.height, src.width)
    bbox = transform_bounds(crs, "EPSG:4326", *bounds)
    cat = pystac_client.Client.open(STAC)

    # ---- choose a thinned, consistent set of scenes ----
    s2_all = search(cat, "sentinel-2-l2a", bbox, query={"eo:cloud_cover": {"lt": 30}})
    best = {}
    for it in s2_all:                                         # one scene per date (lowest cloud)
        d = it.datetime.date(); cc = it.properties.get("eo:cloud_cover", 100)
        if d not in best or cc < best[d].properties.get("eo:cloud_cover", 100): best[d] = it
    s2 = thin(list(best.values()), 9)
    s1_all = search(cat, "sentinel-1-rtc", bbox)
    if s1_all:
        orb = Counter(i.properties.get("sat:orbit_state", "") for i in s1_all).most_common(1)[0][0]
        s1 = thin([i for i in s1_all if i.properties.get("sat:orbit_state", "") == orb], 10)
    else:
        s1 = []

    rows, ok, bad, first = [], 0, 0, ""
    # ---------------- Sentinel-2 ----------------
    for it in s2:
        try:
            d = {k: read_asset(it, k, "sentinel-2-l2a", bounds, crs, shape,
                               Resampling.nearest if k == "SCL" else Resampling.bilinear) for k in S2_ASSETS}
            pb = float(it.properties.get("s2:processing_baseline", "0") or 0)
            off = 1000.0 if pb >= 4.0 else 0.0
            R = {k: np.clip((d[k] - off) / 10000.0, 0, None) for k in S2_ASSETS[:-1]}
            valid = (np.isin(d["SCL"].astype(np.int16), (4, 5, 6, 7)) & (R["B08"] > 0)).ravel()
            F = {"B03": R["B03"], "B04": R["B04"], "B08": R["B08"], "B11": R["B11"],
                 "NDVI": (R["B08"] - R["B04"]) / (R["B08"] + R["B04"] + EPS),
                 "NDMI": (R["B08"] - R["B11"]) / (R["B08"] + R["B11"] + EPS),
                 "NDWI": (R["B03"] - R["B08"]) / (R["B03"] + R["B08"] + EPS)}
            F = {k: v.ravel() for k, v in F.items()}
            date = it.datetime.date().isoformat()
            for split, fid, idx in fields:
                v = valid[idx]
                if v.sum() == 0: continue
                row = dict(chip_id=cid, split=split, field_id=fid, sensor="s2", date=date,
                           n_valid=int(v.sum()), n_px=len(idx))
                for k, a in F.items(): row[k] = float(np.median(a[idx][v]))
                rows.append(row)
            ok += 1
        except Exception as e:
            bad += 1; first = first or repr(e)[:150]
    # ---------------- Sentinel-1 ----------------
    for it in s1:
        try:
            vv = read_asset(it, "vv", "sentinel-1-rtc", bounds, crs, shape, Resampling.bilinear).ravel()
            vh = read_asset(it, "vh", "sentinel-1-rtc", bounds, crs, shape, Resampling.bilinear).ravel()
            valid = np.isfinite(vv) & np.isfinite(vh) & (vv > 0) & (vh > 0)
            with np.errstate(divide="ignore", invalid="ignore"):
                VV, VH = 10 * np.log10(vv), 10 * np.log10(vh)
            date = it.datetime.date().isoformat()
            for split, fid, idx in fields:
                v = valid[idx]
                if v.sum() == 0: continue
                rows.append(dict(chip_id=cid, split=split, field_id=fid, sensor="s1", date=date,
                                 n_valid=int(v.sum()), n_px=len(idx),
                                 VV=float(np.median(VV[idx][v])), VH=float(np.median(VH[idx][v]))))
            ok += 1
        except Exception as e:
            bad += 1; first = first or repr(e)[:150]
    if bad > 0.5 * max(ok + bad, 1):
        raise RuntimeError(f"{cid}: {bad}/{ok+bad} reads failed; first error: {first}")
    pd.to_pickle(pd.DataFrame(rows), out_f)
    return len(rows)

# ---------- background runner: the notebook stays free ----------
PROG = dict(done=0, total=0, errs=[], t0=time.time())
def _bg(chip_list, workers):
    PROG.update(done=0, total=len(chip_list), errs=[], t0=time.time())
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(process_chip, c): c for c in chip_list}
        for f in as_completed(futs):
            try: f.result()
            except Exception as e: PROG["errs"].append((futs[f], repr(e)[:250]))
            PROG["done"] += 1

def start_bg(chip_list, workers=24):
    th = threading.Thread(target=_bg, args=(chip_list, workers), daemon=True); th.start(); return th

def status():
    el, d = time.time() - PROG["t0"], PROG["done"]
    print(f"{d}/{PROG['total']} chips | {el/60:.1f} min | ~{el/max(d,1):.1f} s/chip | "
          f"ETA {(PROG['total']-d)*el/max(d,1)/60:.0f} min | errors: {len(PROG['errs'])}")
    for e in PROG["errs"][:3]: print("  ", e)

In [6]:
RUN_DOWNLOAD = False          # True only if you really want to download missing chips
TS_DIR = CACHE / "ts_fast"
bad = []
for p in TS_DIR.glob("*.pkl"):
    try: pd.read_pickle(p)
    except Exception: bad.append(p)
if bad:
    print(len(bad), "corrupt files deleted"); [p.unlink() for p in bad]
n_cached = len(list(TS_DIR.glob("*.pkl")))
print(f"cached chips: {n_cached} / {len(all_chip_ids)}")
if n_cached == len(all_chip_ids):
    print("Time series complete - nothing to download.")
elif RUN_DOWNLOAD:
    th = start_bg([c for c in all_chip_ids if not (TS_DIR / f"{c}.pkl").exists()], workers=24)
    print("download started in background; run status() to follow it")
else:
    print("Cache incomplete. Set RUN_DOWNLOAD = True to download the missing chips.")

cached chips: 1217 / 1217
Time series complete - nothing to download.
